# MSMARCO-XI Full-Coverage Production RAG Indexer (Punjabi / ਪੰਜਾਬੀ)

This notebook is the full-dataset architecture for the final multilingual RAG system configured for **Punjabi (`pa` / `pan`)**.

**Coverage:** every source record is indexed. `is_selected` is not used to delete records.

**Design:**
`MSMARCO-XI (Punjabi: train/pantrain.parquet) → record-level dense index → candidate records → multi-strategy passage chunking → grounded answer`

Each language is a separate resumable Kaggle job. This prevents one long all-language run from timing out.

Kaggle: Internet ON, GPU ON, HF_TOKEN secret recommended.


## 1. Safe Kaggle environment

In [1]:
import sys, importlib.util, subprocess

required = {
    "huggingface_hub": "huggingface_hub",
    "pyarrow": "pyarrow",
    "transformers": "transformers",
    "faiss": "faiss-cpu",
    "fsspec": "fsspec",
    "tqdm": "tqdm",
}
missing = [pip for mod, pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "-q", *missing])

import numpy as np
import pandas as pd
import scipy
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU REQUIRED. Kaggle Settings → Accelerator → GPU, then restart.")
print("GPU:", torch.cuda.get_device_name(0))
torch.set_float32_matmul_precision("high")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.3 MB/s eta 0:00:00
Python: 3.12.13
NumPy: 2.0.2
Pandas: 2.3.3
SciPy: 1.16.3
Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


## 2. Hugging Face token

In [2]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("HF token available:", bool(HF_TOKEN))

HF token available: False


## 3. Full-coverage configuration

In [3]:
from pathlib import Path

LANGUAGE_NAMES = {
    "as":"Assamese","bn":"Bengali","gu":"Gujarati","hi":"Hindi",
    "kn":"Kannada","ml":"Malayalam","mr":"Marathi","ne":"Nepali",
    "or":"Odia","pa":"Punjabi","sa":"Sanskrit","ta":"Tamil",
    "te":"Telugu","ur":"Urdu",
}
LANGUAGE_PREFIXES = {
    "as":"asm","bn":"ben","gu":"guj","hi":"hin","kn":"kan",
    "ml":"mal","mr":"mar","ne":"nep","or":"ori","pa":"pan",
    "sa":"san","ta":"tam","te":"tel","ur":"urd",
}

LANGUAGE = "pa"          # change for each Kaggle job
SPLIT = "train"
PASSAGE_MODE = "translated"

FULL_RUN = True           # ALL rows for this language
RECORD_TEXT_MAX_CHARS = 7000

# Large-scale index: approximately one vector per source record.
FAISS_NLIST = 4096
FAISS_PQ_M = 48
FAISS_PQ_BITS = 8
FAISS_TRAIN_RECORDS = 200_000

# Candidate-stage chunking options
FIXED_SIZE = 500
FIXED_OVERLAP = 80
SENTENCES_PER_CHUNK = 3
SEMANTIC_THRESHOLD = 0.58

BATCH_ROWS = 1024
EMBED_BATCH_SIZE = 512
ROWS_PER_RECORD_SHARD = 100_000

MODEL_NAME = "intfloat/multilingual-e5-small"

ROOT = Path("/kaggle/working/msmarco_xi_full")
LANG_ROOT = ROOT / LANGUAGE
RECORD_ROOT = LANG_ROOT / "records"
RECORD_ROOT.mkdir(parents=True, exist_ok=True)

print("Language:", LANGUAGE_NAMES[LANGUAGE])
print("FULL RUN:", FULL_RUN)

Language: Bengali
FULL RUN: True


## 4. Open the exact remote Parquet file

In [4]:
from huggingface_hub import HfApi, hf_hub_url
import fsspec
import pyarrow.parquet as pq

repo_file = f"{SPLIT}/{LANGUAGE_PREFIXES[LANGUAGE]}train.parquet"
if SPLIT != "train":
    repo_file = f"{SPLIT}/{LANGUAGE_PREFIXES[LANGUAGE]}val.parquet"

api = HfApi(token=HF_TOKEN)
info = api.repo_info(repo_id="ai4bharat/MSMARCO-XI", repo_type="dataset")

url = hf_hub_url(
    repo_id="ai4bharat/MSMARCO-XI",
    filename=repo_file,
    repo_type="dataset",
    revision="main",
)

headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
fs = fsspec.filesystem("https", headers=headers)
remote_handle = fs.open(url, "rb", block_size=16 * 1024 * 1024, cache_type="readahead")
parquet_file = pq.ParquetFile(remote_handle)

SOURCE_ROWS = int(parquet_file.metadata.num_rows)
print("File:", repo_file)
print("Source rows:", f"{SOURCE_ROWS:,}")

File: train/bentrain.parquet
Source rows: 778,638


## 5. GPU embedding model

In [5]:
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
).to(device)
model.eval()

@torch.inference_mode()
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).expand(hidden.size()).float()
    return (hidden * m).sum(1) / torch.clamp(m.sum(1), min=1e-9)

@torch.inference_mode()
def encode_texts(texts, prefix="passage: "):
    if not texts:
        return np.empty((0, model.config.hidden_size), dtype="float32")
    all_vecs = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = [prefix + str(x) for x in texts[i:i+EMBED_BATCH_SIZE]]
        tokens = tokenizer(
            batch, padding=True, truncation=True, max_length=384, return_tensors="pt"
        )
        tokens = {k: v.to(device, non_blocking=True) for k, v in tokens.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(**tokens)
            emb = mean_pool(out.last_hidden_state, tokens["attention_mask"])
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_vecs.append(emb.float().cpu().numpy().astype("float32"))
    return np.vstack(all_vecs)

print("Embedding test:", encode_texts(["test"]).shape)

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding test: (1, 384)


## 6. Record representation: one vector per source record, all passages preserved

In [6]:
def build_record_text(record):
    passages = record.get("passages") or {}
    if PASSAGE_MODE == "translated":
        texts = passages.get("Translated_passages") or []
    else:
        texts = passages.get("English_passages") or []

    selected = passages.get("is_selected") or []
    selected_texts = [
        str(texts[i]).strip()
        for i, flag in enumerate(selected)
        if flag == 1 and i < len(texts) and str(texts[i]).strip()
    ]
    if not selected_texts:
        selected_texts = [str(x).strip() for x in texts[:2] if str(x).strip()]

    pieces = [
        str(record.get("query", "")).strip(),
        str(record.get("Answer", "")).strip(),
        *selected_texts,
    ]
    return "\n".join(x for x in pieces if x)[:RECORD_TEXT_MAX_CHARS]

def normalize_record(record, local_id):
    passages = record.get("passages") or {}
    return {
        "local_id": int(local_id),
        "query_id": int(record.get("query_id", 0)),
        "query": str(record.get("query", "")),
        "answer": str(record.get("Answer", "")),
        "query_type": str(record.get("query_type", "")),
        "source_lang": str(record.get("source_lang", "")),
        "target_lang": str(record.get("target_lang", "")),
        "english_passages": [str(x) for x in passages.get("English_passages") or []],
        "translated_passages": [str(x) for x in passages.get("Translated_passages") or []],
        "is_selected": [int(x) for x in passages.get("is_selected") or []],
    }

## 7. Train IVF-PQ from a sample

In [7]:
import faiss

DIM = int(model.config.hidden_size)

train_texts = []
for batch in parquet_file.iter_batches(
    batch_size=2048,
    columns=["query","Answer","passages"],
):
    for record in batch.to_pylist():
        text = build_record_text(record)
        if text:
            train_texts.append(text)
        if len(train_texts) >= FAISS_TRAIN_RECORDS:
            break
    if len(train_texts) >= FAISS_TRAIN_RECORDS:
        break

train_vectors = encode_texts(train_texts)

quantizer = faiss.IndexFlatIP(DIM)
index = faiss.IndexIVFPQ(
    quantizer, DIM, FAISS_NLIST, FAISS_PQ_M, FAISS_PQ_BITS,
    faiss.METRIC_INNER_PRODUCT
)
index.train(train_vectors)

print("IVF-PQ trained.")
print("Training records:", len(train_texts))
print("nlist:", FAISS_NLIST, "PQ:", f"{FAISS_PQ_M}x{FAISS_PQ_BITS}")

IVF-PQ trained.
Training records: 200000
nlist: 4096 PQ: 48x8


## 8. Compressed source-record store

In [8]:
import pyarrow as pa
import pyarrow.parquet as pq

schema = pa.schema([
    ("local_id", pa.int64()),
    ("query_id", pa.int64()),
    ("query", pa.string()),
    ("answer", pa.string()),
    ("query_type", pa.string()),
    ("source_lang", pa.string()),
    ("target_lang", pa.string()),
    ("english_passages", pa.list_(pa.string())),
    ("translated_passages", pa.list_(pa.string())),
    ("is_selected", pa.list_(pa.int8())),
])

def write_shard(rows, shard_id):
    if not rows:
        return
    path = RECORD_ROOT / f"records_{shard_id:05d}.parquet"
    table = pa.Table.from_pydict(
        {k: [r[k] for r in rows] for k in schema.names},
        schema=schema,
    )
    pq.write_table(
        table, path,
        compression="zstd",
        compression_level=7,
        use_dictionary=True,
    )

## 9. Full indexing loop with checkpoint/resume

In [9]:
import json
import time
from tqdm.auto import tqdm

INDEX_PATH = LANG_ROOT / "faiss_ivfpq.index"
CHECKPOINT_PATH = LANG_ROOT / "checkpoint.json"
CONFIG_PATH = LANG_ROOT / "config.json"

next_id = 0

if INDEX_PATH.exists() and CHECKPOINT_PATH.exists():
    index = faiss.read_index(str(INDEX_PATH))
    cp = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    next_id = int(cp["next_id"])
    print("Resuming at local_id:", next_id)

pending_texts, pending_ids, pending_records = [], [], []
shard_rows = []
shard_id = next_id // ROWS_PER_RECORD_SHARD
started = time.perf_counter()

def flush():
    global pending_texts, pending_ids, pending_records, shard_rows, shard_id, next_id

    if not pending_texts:
        return

    vecs = encode_texts(pending_texts, prefix="passage: ")
    ids = np.asarray(pending_ids, dtype=np.int64)
    index.add_with_ids(vecs, ids)

    shard_rows.extend(pending_records)

    while len(shard_rows) >= ROWS_PER_RECORD_SHARD:
        write_shard(shard_rows[:ROWS_PER_RECORD_SHARD], shard_id)
        del shard_rows[:ROWS_PER_RECORD_SHARD]
        shard_id += 1

    pending_texts.clear()
    pending_ids.clear()
    pending_records.clear()

    next_id = int(ids[-1]) + 1
    faiss.write_index(index, str(INDEX_PATH))
    CHECKPOINT_PATH.write_text(
        json.dumps({"next_id": next_id, "vectors": int(index.ntotal)}, indent=2),
        encoding="utf-8",
    )

for batch in tqdm(
    parquet_file.iter_batches(
        batch_size=BATCH_ROWS,
        columns=[
            "source_lang","target_lang","Answer","query_id",
            "query_type","passages","query"
        ],
    ),
    desc=f"FULL {LANGUAGE_NAMES[LANGUAGE]}",
):
    for record in batch.to_pylist():
        pending_texts.append(build_record_text(record))
        pending_ids.append(next_id)
        pending_records.append(normalize_record(record, next_id))
        next_id += 1

        if len(pending_texts) >= EMBED_BATCH_SIZE:
            flush()

flush()

if shard_rows:
    write_shard(shard_rows, shard_id)

faiss.write_index(index, str(INDEX_PATH))

elapsed = time.perf_counter() - started
CONFIG_PATH.write_text(
    json.dumps({
        "dataset": "ai4bharat/MSMARCO-XI",
        "language": LANGUAGE,
        "source_file": repo_file,
        "source_rows": SOURCE_ROWS,
        "records_indexed": int(index.ntotal),
        "all_records": True,
        "all_passages_preserved": True,
        "index": "IVFPQ",
        "dimension": DIM,
        "nlist": FAISS_NLIST,
        "pq_m": FAISS_PQ_M,
        "pq_bits": FAISS_PQ_BITS,
        "embedding_model": MODEL_NAME,
        "elapsed_seconds": elapsed,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("FULL COMPLETE")
print("Source rows:", f"{SOURCE_ROWS:,}")
print("Vectors:", f"{index.ntotal:,}")
print("Elapsed:", round(elapsed, 1), "sec")

FULL Bengali: 0it [00:00, ?it/s]

FULL COMPLETE
Source rows: 778,638
Vectors: 778,638
Elapsed: 2560.3 sec


## 10. Candidate-stage chunking

All records are preserved, but the system does not need four giant vector indexes.

After record retrieval, the backend can apply:
- fixed-size + overlap
- sentence-aware
- semantic
- metadata-aware

to the passages of the top candidate records.

In [10]:
import re

def fixed_chunks(text, size=FIXED_SIZE, overlap=FIXED_OVERLAP):
    text = str(text).strip()
    out, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if text[start:end].strip():
            out.append(text[start:end].strip())
        if end >= len(text):
            break
        start += size - overlap
    return out

def sentence_chunks(text):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?।॥])\s+", str(text)) if s.strip()]
    return [
        " ".join(sentences[i:i + SENTENCES_PER_CHUNK])
        for i in range(0, len(sentences), SENTENCES_PER_CHUNK)
    ]

def metadata_aware_chunks(text, query_type, language):
    return [
        f"[type={query_type} language={language}] {chunk}"
        for chunk in sentence_chunks(text)
    ]

def semantic_chunks(text):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?।॥])\s+", str(text)) if s.strip()]
    if len(sentences) <= 1:
        return sentences
    vectors = encode_texts(sentences)
    out, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        if float(np.dot(vectors[i-1], vectors[i])) < SEMANTIC_THRESHOLD:
            out.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])
    if current:
        out.append(" ".join(current))
    return out

## 11. Retrieval benchmark

This benchmark is retrieval-only. The final RAG dashboard must separately measure:
STT, query embedding, FAISS, record lookup, chunk/rerank, guardrails, LLM TTFT, LLM total, and E2E total.

Do not claim the retrieval number is the full 200 ms result.

In [11]:
final_index = faiss.read_index(str(INDEX_PATH))

def retrieve(query, top_k=20):
    q = encode_texts([query], prefix="query: ")
    scores, ids = final_index.search(q, top_k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]

queries = []
for batch in parquet_file.iter_batches(batch_size=512, columns=["query"]):
    for record in batch.to_pylist():
        q = str(record.get("query", "")).strip()
        if q:
            queries.append(q)
        if len(queries) >= 100:
            break
    if len(queries) >= 100:
        break

times = []
for q in queries:
    t0 = time.perf_counter()
    retrieve(q)
    times.append((time.perf_counter() - t0) * 1000)

arr = np.asarray(times)
print("Queries:", len(arr))
print(f"P50: {np.percentile(arr,50):.2f} ms")
print(f"P70: {np.percentile(arr,70):.2f} ms")
print(f"P100: {np.max(arr):.2f} ms")

Queries: 100
P50: 9.95 ms
P70: 10.14 ms
P100: 53.63 ms


# 12. LLM latency fix for your current 5-second pipeline

Your current screenshot shows approximately:

`STT ≈ 2425 ms`
`FAISS ≈ 52 ms`
`Guard ≈ 1 ms`
`LLM ≈ 2594 ms`

The large LLM is therefore a major bottleneck.

For the final backend:

- Keep the model warm (`keep_alive=-1`).
- Use a small quantized model.
- Disable thinking/reasoning for short grounded QA.
- Send only 2–4 chunks to the LLM.
- Keep context around 2K tokens.
- Cap output to 32–48 tokens.
- `temperature=0`.
- Stream output for lower perceived latency.
- Benchmark on the actual deployment machine; do not claim <200 ms without measurement.

Good first Ollama candidates:
- `gemma3:1b` for minimum latency.
- `qwen3:4b` for a quality/speed compromise.

Ollama currently lists Gemma 3 1B as a small model and Gemma 3 4B at about 3.3 GB; Qwen3 4B is about 2.5 GB in Q4_K_M. citeturn319747search1turn319747search0turn319747search2

In [12]:
llm_config = {
    "first_test_model": "gemma3:1b",
    "alternative_model": "qwen3:4b",
    "keep_alive": -1,
    "stream": True,
    "temperature": 0,
    "num_ctx": 2048,
    "num_predict": 48,
    "top_k_context": 3,
    "grounded_only": True,
}

( ROOT / "llm_latency_config.json" ).write_text(
    json.dumps(llm_config, indent=2),
    encoding="utf-8",
)
print("LLM config:", ROOT / "llm_latency_config.json")

LLM config: /kaggle/working/msmarco_xi_full/llm_latency_config.json


## 13. Final artifact layout

```text
<language>/
├── faiss_ivfpq.index
├── config.json
├── checkpoint.json
└── records/
    ├── records_00000.parquet
    ├── records_00001.parquet
    └── ...
```

**FAISS** is the fast candidate-retrieval layer.

**Parquet record shards** preserve every source record and all its passages.

Do not push these large artifacts into GitHub.